# Phi-3 Medium Demo - Hadoop vs Spark Comparison

Notebook nay chay tren Google Colab, mount Google Drive de dung model Phi-3 Medium (nap duoi dang 4-bit de tiet kiem GPU VRAM) de tra loi mot cau hoi duy nhat phan tich va so sanh Hadoop vs Apache Spark bang tieng Viet voi cau hinh System Prompt chuyen gia Big Data.

## 1. Environment and Google Drive Setup

Mount Google Drive de su dung Hugging Face cache co san, clone hoac pull ma nguon moi nhat tu GitHub va thiet lap thu muc lam viec.

In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
REPO_BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive")

if IN_COLAB:
    REPO_DIR = Path("/content/a-triple-of-lms")
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    # Tu dong phat hien thu muc root du an locally
    cwd = Path.cwd()
    if (cwd / "src" / "phi3.py").exists():
        REPO_DIR = cwd
    elif (cwd.parent / "src" / "phi3.py").exists():
        REPO_DIR = cwd.parent
    else:
        REPO_DIR = cwd

if not (REPO_DIR / "src" / "phi3.py").exists():
    raise FileNotFoundError(f"Repo/script not found: {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print("IN_COLAB:", IN_COLAB)
print("Repo:", REPO_DIR)
print("Drive:", DRIVE_ROOT if IN_COLAB else "not mounted")

## 2. Install Dependencies

Cai dat cac goi thu vien can thiet de nap va chay model cuc bo tren Colab GPU.

In [ ]:
# Install runtime dependencies. Safe to rerun.
packages = [
    "transformers>=4.41.0",
    "accelerate>=0.30.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "einops",
    "tqdm",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)
print("Dependencies installed")

## 3. Configuration and Prompt Setup

Cau hinh cac tham so cho model va thiet lap he thong System Prompt va User Prompt bang tieng Viet theo dung yeu cau.

In [ ]:
# Folder chua Hugging Face cache tren Drive
HF_CACHE_DIR = DRIVE_ROOT / "hf_cache"

LOCAL_FILES_ONLY = True       # True = chi lay tu cache tren Drive. Chuyen thanh False neu muon tai truc tiep tu Hugging Face
LOAD_IN_4BIT = True           # Nap duoi dang 4-bit de phu hop voi Colab GPU
DEVICE = "auto"
TORCH_DTYPE = "float16"

MODEL_ID = "microsoft/Phi-3-medium-4k-instruct"

SYSTEM_PROMPT_VN = (
    "Bạn là một chuyên gia Big Data Engineer có kinh nghiệm triển khai hệ thống xử lý dữ liệu lớn trong môi trường production. "
    "Hãy trả lời chính xác, dễ hiểu, có cấu trúc rõ ràng và liên hệ với các use case thực tế. Khi so sánh công nghệ, "
    "cần nêu rõ kiến trúc, ưu điểm, hạn chế, trường hợp nên sử dụng và trade-off khi triển khai."
)

USER_PROMPT = """Hãy trình bày sự khác nhau giữa Hadoop và Apache Spark trong xử lý dữ liệu lớn.

Yêu cầu câu trả lời bằng tiếng Việt, có cấu trúc rõ ràng theo các phần sau:

1. Tổng quan ngắn gọn về Hadoop và Apache Spark.
2. So sánh hai công nghệ theo các tiêu chí:
     - Kiến trúc xử lý dữ liệu
     - Cơ chế lưu trữ và tính toán
     - Hiệu năng
     - Khả năng xử lý batch, streaming và interactive analytics
     - Mức độ phù hợp trong các hệ thống Big Data hiện đại
3. Trình bày ưu điểm và hạn chế của Hadoop.
4. Trình bày ưu điểm và hạn chế của Apache Spark.
5. Cho ví dụ thực tế: khi nào nên dùng Hadoop, khi nào nên dùng Spark.
6. Kết luận ngắn gọn: nếu xây dựng một pipeline phân tích dữ liệu lớn hiện nay, nên chọn công nghệ nào trong từng trường hợp.

Yêu cầu thêm:
    - Trả lời dễ hiểu cho sinh viên mới học Big Data.
    - Không chỉ liệt kê, hãy giải thích ý nghĩa của từng điểm khác biệt.
    - Có thể dùng bảng so sánh nếu phù hợp.
    - Không trả lời quá dài, ưu tiên rõ ràng và thực tế."""

## 4. Run Generation

Nap model su dung logic cua phi3.py va sinh cau tra loi duy nhat.

In [ ]:
from src.phi3 import (
    normalize_hub_cache_dir,
    run_single_inference,
)

cache_dir = normalize_hub_cache_dir(str(HF_CACHE_DIR)) if HF_CACHE_DIR else None
print("Resolved cache_dir:", cache_dir)

print("Starting inference (it may take a few minutes)...\n")
response = run_single_inference(
    model_id_or_path=MODEL_ID,
    prompt=USER_PROMPT,
    system_prompt=SYSTEM_PROMPT_VN,
    cache_dir=cache_dir,
    local_files_only=LOCAL_FILES_ONLY,
    device=DEVICE,
    torch_dtype_name=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    trust_remote_code=False,
    temperature=0.2,
    max_new_tokens=4000,
)

## 5. Render Response

Hien thi cau tra loi co dinh dang Markdown hoan chinh va dep mat.

In [ ]:
from IPython.display import display, Markdown

print("Cau tra loi tu Phi-3 Medium:\n")
display(Markdown(response))